In [1]:
# In[1]:


# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load in 

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the "../input/" directory.
# For example, running this (by clicking run or pressing Shift+Enter) will list the files in the input directory

import os
print((os.listdir("../input")))

# Any results you write to the current directory are saved as output.

In [2]:
# In[2]:


train_df=pd.read_csv("../input/train.csv")
test_df=pd.read_csv("../input/test.csv")

In [3]:
# In[3]:


print((train_df.info()))
print(("--"*20))
print((test_df.info()))

In [4]:
# In[4]:


X=train_df.drop("Survived",axis=1)
Y=train_df["Survived"]

In [5]:
# In[5]:


combined_df=pd.concat([X,test_df])

In [6]:
# In[6]:


print((combined_df.info()))


# # Data Cleaning
# * Correct
# * Complete
# * Create
# * Convert

In [7]:
# In[7]:


from sklearn.impute import SimpleImputer

Age_imputer=SimpleImputer(strategy="median")
Fare_imputer=SimpleImputer(strategy="median")
Embarked_imputer=SimpleImputer(strategy="most_frequent")


# The imputers are no longer fitted here on the combined train+test data.
# They are fitted on training data only, as steps of the `preprocessor`
# pipeline defined below (In[12]), applied after the train/test split.

In [8]:
# In[8]:


combined_df.drop(["Cabin"],axis=1,inplace=True)

In [9]:
# In[9]:


combined_df["Family_size"]=combined_df["SibSp"]+combined_df["Parch"]+1
combined_df["Is_alone"]=int(0)
combined_df["Is_alone"][combined_df["Family_size"]==1]=int(1)

In [10]:
# In[10]:


combined_df['Title']=combined_df['Name'].str.split(",", expand=True)[1].str.split(".", expand=True)[0]

In [11]:
# In[11]:


import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin

class Binner(BaseEstimator, TransformerMixin):
    """pd.qcut / pd.cut whose bin edges are learned on the fitted (training) data only.
    Outputs the integer bin code, i.e. what LabelEncoder gave on the sorted bins."""
    def __init__(self, kind="qcut", bins=4):
        self.kind = kind
        self.bins = bins
    def fit(self, X, y=None):
        x = np.asarray(X, dtype=float).ravel()
        cut = pd.qcut if self.kind == "qcut" else pd.cut
        _, edges = cut(x, self.bins, retbins=True)
        self.edges_ = np.concatenate([[-np.inf], edges[1:-1], [np.inf]])
        return self
    def transform(self, X):
        x = np.asarray(X, dtype=float).ravel()
        return pd.cut(x, self.edges_, labels=False).reshape(-1, 1)

# Fare_bin = qcut(Fare, 4) and Age_bin = cut(Age.astype(int), 4), fitted on training data only
Fare_binner=Binner(kind="qcut", bins=4)
Age_binner=Binner(kind="cut", bins=4)

In [12]:
# In[12]:


from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
encoder=LabelEncoder()

# Imputation, binning and encoding of the model features are fitted on training
# data only (and re-fitted inside every CV fold) via this preprocessing pipeline.
# Output column order: Pclass, Family_size, Is_alone, Sex_encode, Age_encode, Fare_encode, Embarked_encode
preprocessor=ColumnTransformer([
    ("Pclass","passthrough",["Pclass"]),
    ("Family_size","passthrough",["Family_size"]),
    ("Is_alone","passthrough",["Is_alone"]),
    ("Sex_encode",OrdinalEncoder(handle_unknown="use_encoded_value",unknown_value=-1),["Sex"]),
    ("Age_encode",Pipeline([("impute",Age_imputer),("to_int",FunctionTransformer(np.trunc)),("bin",Age_binner)]),["Age"]),
    ("Fare_encode",Pipeline([("impute",Fare_imputer),("bin",Fare_binner)]),["Fare"]),
    ("Embarked_encode",Pipeline([("impute",Embarked_imputer),("encode",OrdinalEncoder(handle_unknown="use_encoded_value",unknown_value=-1))]),["Embarked"]),
])

# Title_encode is not used as a model feature
combined_df["Title_encode"]=encoder.fit_transform(combined_df["Title"])

In [13]:
# In[13]:


combined_df.head()

In [14]:
# In[14]:


# Features to be used (raw columns; imputed/binned/encoded by `preprocessor` after the split)
df_final=combined_df[["Pclass","Family_size","Is_alone","Sex","Age","Fare","Embarked"]]
X_train=df_final[0:891]
X_test=df_final[891:]

In [15]:
# In[15]:


X_train.head()


# # Model Creation

In [16]:
# In[16]:


from sklearn import ensemble, neural_network , neighbors , svm , linear_model ,naive_bayes ,tree , gaussian_process,discriminant_analysis
from xgboost import XGBClassifier, XGBRFClassifier

In [17]:
# In[17]:


#Machine Learning Algorithms
MLA=[
    #Ensemble Methods
    ensemble.AdaBoostClassifier(),
    ensemble.BaggingClassifier(),
    ensemble.ExtraTreesClassifier(),
    ensemble.GradientBoostingClassifier(),
    ensemble.RandomForestClassifier(),

    #Gaussian Processes
    gaussian_process.GaussianProcessClassifier(),
    
    #GLM
    linear_model.LogisticRegressionCV(),
    linear_model.PassiveAggressiveClassifier(),
    linear_model.RidgeClassifierCV(),
    linear_model.SGDClassifier(),
    linear_model.Perceptron(),
    
    #Navies Bayes
    naive_bayes.BernoulliNB(),
    naive_bayes.GaussianNB(),
    
    #Nearest Neighbor
    neighbors.KNeighborsClassifier(),
    
    #SVM
    svm.SVC(probability=True),
    svm.NuSVC(probability=True),
    svm.LinearSVC(),
    
    #Trees    
    tree.DecisionTreeClassifier(),
    tree.ExtraTreeClassifier(),
    
    #Discriminant Analysis
    discriminant_analysis.LinearDiscriminantAnalysis(),
    discriminant_analysis.QuadraticDiscriminantAnalysis(),

    #Neural Network
    neural_network.MLPClassifier(),
    
    #xgboost
    XGBClassifier(),
    XGBRFClassifier()
]

In [18]:
# In[18]:


MLA_compare=["Algorithm_Name","Parameters", "Mean_Accuracy","Standard_Deviation"]
MLA_compare=pd.DataFrame(columns=MLA_compare)

In [19]:
# In[19]:


from sklearn.model_selection import cross_val_score
from sklearn.base import clone

index=0
for algo in MLA:
    clf=Pipeline([("preprocess",clone(preprocessor)),("model",algo)])
    clf.fit(X_train,Y)
    cv=cross_val_score(estimator=clf,X=X_train,y=Y,cv=10,n_jobs=-1)
    cv_mean=cv.mean()
    cv_std=cv.std()
    MLA_compare.loc[index,"Algorithm_Name"]=algo.__class__.__name__
    MLA_compare.loc[index,"Parameters"]=str(algo.get_params())
    MLA_compare.loc[index,"Mean_Accuracy"]=cv_mean
    MLA_compare.loc[index,"Standard_Deviation"]=cv_std
    index+=1

In [20]:
# In[20]:


MLA_compare


# ## Fine Tuning Models by tuning Hyper parameters

In [21]:
# In[21]:


selected_model=Pipeline([("preprocess",clone(preprocessor)),("model",svm.NuSVC(probability=True))])
selected_model.fit(X_train,Y)

In [22]:
# In[22]:


# Parameters=[{"base_score":[0.1,0.2,0.3,0.5,0.6,0.9,1,2],
#              "gamma":[0,0.1,0.2,0.3,0.4,0.5,1],
#              "learning_rate":[0,0.1,0.2,0.3,0.4,1],
#             }]

In [23]:
# In[23]:


# from sklearn.model_selection import GridSearchCV
# grid=GridSearchCV(estimator=selected_model,param_grid=Parameters,n_jobs=-1,cv=10,scoring="accuracy")
# result=grid.fit(X=X_train,y=Y)

In [24]:
# In[24]:


# best_accuracy=result.best_score_
# best_parameters=result.best_params_

In [25]:
# In[25]:


y_pred=selected_model.predict(X_test)
y_pred=pd.DataFrame({"PassengerId":test_df["PassengerId"],"Survived":y_pred})
y_pred.to_csv("Submit.csv",index=False)

In [26]:
# In[26]:


test_df

In [ ]:
# In[ ]:

In [ ]:
# In[ ]: